# Full test-set evaluation: ResUNet and data-consistency post-processing

## Prepare

In [ ]:
from src.general_utils import prepare_environment
from mri_dl import (
    MRIUndersampledDataset,
    ResidualUNet,
    load_checkpoint,
    predict_numpy_with_data_consistency,
)
from src.metrices import calculate_psnr, calculate_ssim
import os
from pathlib import Path
import numpy as np
import pandas as pd

HPC = False
prepare_environment(hpc=HPC)

## Load test dataset and trained model

In [ ]:
data_sets_root = Path(os.getcwd() + r"/../reconstruction_dataset").resolve()

test_dataset = MRIUndersampledDataset(
    data_sets_root / "test",
    planes=("Coronal",),
    retain_ratios=(0.30,),
)

loaded_model, checkpoint = load_checkpoint(
    checkpoint_path="checkpoints/coronal_r30_delta_best.pt",
    model_class=ResidualUNet,
)

print(f"Test samples: {len(test_dataset)}")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["validation_loss"])

## Evaluate the full test set

In [ ]:
if len(test_dataset) == 0:
    raise ValueError("test_dataset is empty")

records = []
for idx in range(len(test_dataset)):
    sample = test_dataset[idx]
    row = test_dataset.samples.iloc[idx]

    if "k_space_file" not in row.index:
        raise KeyError("samples.csv is missing 'k_space_file'. Recreate dataset with notebook 05.")

    k_space_path = test_dataset.split_root / str(row["k_space_file"])
    if not k_space_path.exists():
        raise FileNotFoundError(f"K-space file not found: {k_space_path}")

    acquired_k_space = np.load(k_space_path, allow_pickle=False)
    target = sample["target"][0].numpy()
    undersampled = sample["input"][0].numpy()

    post = predict_numpy_with_data_consistency(
        model=loaded_model,
        image=undersampled,
        acquired_k_space=acquired_k_space,
        return_debug=True,
    )
    cnn_reconstruction = post["reconstructed_image"]
    final_reconstruction = post["final_image"]

    records.append(
        {
            "sample_index": idx,
            "undersampled_psnr": calculate_psnr(target, undersampled),
            "cnn_psnr": calculate_psnr(target, cnn_reconstruction),
            "final_psnr": calculate_psnr(target, final_reconstruction),
            "undersampled_ssim": calculate_ssim(target, undersampled),
            "cnn_ssim": calculate_ssim(target, cnn_reconstruction),
            "final_ssim": calculate_ssim(target, final_reconstruction),
        }
    )

per_sample_df = pd.DataFrame(records)
print(f"Evaluated {len(per_sample_df)} / {len(test_dataset)} samples")

## Summary table (mean/std)

In [ ]:
summary_rows = [
    {
        "Method": "undersampled image",
        "Mean PSNR": per_sample_df["undersampled_psnr"].mean(),
        "STD PSNR": per_sample_df["undersampled_psnr"].std(ddof=1),
        "Mean SSIM": per_sample_df["undersampled_ssim"].mean(),
        "STD SSIM": per_sample_df["undersampled_ssim"].std(ddof=1),
    },
    {
        "Method": "ResUnet results",
        "Mean PSNR": per_sample_df["cnn_psnr"].mean(),
        "STD PSNR": per_sample_df["cnn_psnr"].std(ddof=1),
        "Mean SSIM": per_sample_df["cnn_ssim"].mean(),
        "STD SSIM": per_sample_df["cnn_ssim"].std(ddof=1),
    },
    {
        "Method": "ResUnet+data consistency (post processing) results",
        "Mean PSNR": per_sample_df["final_psnr"].mean(),
        "STD PSNR": per_sample_df["final_psnr"].std(ddof=1),
        "Mean SSIM": per_sample_df["final_ssim"].mean(),
        "STD SSIM": per_sample_df["final_ssim"].std(ddof=1),
    },
]

summary_df = pd.DataFrame(summary_rows).set_index("Method")
display(summary_df.style.format(precision=4))

## Average deltas per step

In [ ]:
delta_df = pd.DataFrame(
    [
        {
            "Step": "delta_CNN (ResUnet - undersampled)",
            "Mean Delta PSNR": (per_sample_df["cnn_psnr"] - per_sample_df["undersampled_psnr"]).mean(),
            "Mean Delta SSIM": (per_sample_df["cnn_ssim"] - per_sample_df["undersampled_ssim"]).mean(),
        },
        {
            "Step": "delta_consistency (ResUnet+DC - ResUnet)",
            "Mean Delta PSNR": (per_sample_df["final_psnr"] - per_sample_df["cnn_psnr"]).mean(),
            "Mean Delta SSIM": (per_sample_df["final_ssim"] - per_sample_df["cnn_ssim"]).mean(),
        },
    ]
).set_index("Step")

display(delta_df.style.format(precision=4))


## Post-processing outcome counts

In [ ]:
total = len(per_sample_df)

psnr_improved = per_sample_df["final_psnr"] > per_sample_df["cnn_psnr"]
ssim_improved = per_sample_df["final_ssim"] > per_sample_df["cnn_ssim"]
psnr_up_ssim_down = (per_sample_df["final_psnr"] > per_sample_df["cnn_psnr"]) & (per_sample_df["final_ssim"] < per_sample_df["cnn_ssim"])

outcome_df = pd.DataFrame(
    [
        {
            "Condition": "Post processing improved PSNR",
            "Count": int(psnr_improved.sum()),
            "Percentage (%)": 100.0 * float(psnr_improved.mean()),
        },
        {
            "Condition": "Post processing improved SSIM",
            "Count": int(ssim_improved.sum()),
            "Percentage (%)": 100.0 * float(ssim_improved.mean()),
        },
        {
            "Condition": "PSNR improved while SSIM reduced",
            "Count": int(psnr_up_ssim_down.sum()),
            "Percentage (%)": 100.0 * float(psnr_up_ssim_down.mean()),
        },
    ]
).set_index("Condition")

print(f"Total evaluated images: {total}")
display(outcome_df.style.format({"Count": "{:.0f}", "Percentage (%)": "{:.2f}"}))

